---
**⚠️ NOTEBOOK DE TEST UNIQUEMENT**

**CE NOTEBOOK N'EST PAS UNE ANALYSE**  
C'est un notebook de **vérification/setup** uniquement.

Les données affichées ici sont des **échantillons de test** et ne servent qu'à valider:
- Installation des packages
- Connexion SQL Server  
- Accès aux fichiers CSV
- Configuration du projet

**Pour l'analyse réelle, voir:** `01_exploration_sql.ipynb` et `02_exploration_csv.ipynb`

---

# 00 - Setup et Tests

Ce notebook vérifie l'installation de l'environnement et teste les connexions aux sources de données.

## 1. Import des packages

In [10]:
# Imports standard
import os
import sys
from pathlib import Path

# Imports data science
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Imports SQL
import pymssql
from sqlalchemy import create_engine

# Imports configuration
import yaml
from dotenv import load_dotenv

print("✅ Tous les packages importés avec succès")

✅ Tous les packages importés avec succès


## 2. Chargement de la configuration

In [11]:
# Chemin du projet
PROJECT_ROOT = Path("/home/esprit/airlLines_Project")
CONFIG_FILE = PROJECT_ROOT / "config" / "config.yaml"

# Chargement config YAML
with open(CONFIG_FILE, 'r') as f:
    config = yaml.safe_load(f)

print(f"📁 Projet: {PROJECT_ROOT}")
print(f"⚙️  SQL Server: {config['sql']['server']}:{config['sql']['port']}")
print(f"🗄️  Base DWH: {config['sql']['databases']['dwh']}")
print(f"🗄️  Base STG: {config['sql']['databases']['stg']}")

📁 Projet: /home/esprit/airlLines_Project
⚙️  SQL Server: 192.168.1.202:1433
🗄️  Base DWH: airlines_dwh
🗄️  Base STG: airlines_stg


## 3. Test connexion SQL Server

In [12]:
# Configuration SQL
sql_config = config['sql']

# Test connexion directe
try:
    conn = pymssql.connect(
        server=sql_config['server'],
        port=sql_config['port'],
        user=sql_config['user'],
        password=sql_config['password'],
        database='master',
        timeout=sql_config['timeout']
    )
    print("✅ Connexion SQL Server réussie")
    
    # Liste des bases
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sys.databases WHERE name IN ('airlines_dwh', 'airlines_stg')")
    dbs = cursor.fetchall()
    print(f"📊 Bases disponibles: {[db[0] for db in dbs]}")
    
    conn.close()
except Exception as e:
    print(f"❌ Erreur de connexion SQL: {e}")

✅ Connexion SQL Server réussie
📊 Bases disponibles: ['airlines_dwh', 'airlines_stg']


## 4. Création des dossiers de données

In [13]:
# Création des dossiers
folders = [
    PROJECT_ROOT / config['paths']['raw_dir'],
    PROJECT_ROOT / config['paths']['processed_dir'],
    PROJECT_ROOT / config['paths']['features_dir'],
    PROJECT_ROOT / config['paths']['models_dir'],
    PROJECT_ROOT / config['paths']['reports_dir']
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ Tous les dossiers créés/vérifiés")
for folder in folders:
    print(f"   {folder}")

✅ Tous les dossiers créés/vérifiés
   /home/esprit/airlLines_Project/data/raw
   /home/esprit/airlLines_Project/data/processed
   /home/esprit/airlLines_Project/data/features
   /home/esprit/airlLines_Project/models
   /home/esprit/airlLines_Project/reports


## 5. Vérification des fichiers CSV

In [14]:
# Liste des fichiers CSV
csv_dir = PROJECT_ROOT / "dataSetAirlines"
csv_files = list(csv_dir.glob("*.csv"))

print(f"📂 Dossier CSV: {csv_dir}")
print(f"📄 Nombre de fichiers: {len(csv_files)}")
print()
print("Fichiers disponibles:")
for csv_file in sorted(csv_files):
    size = csv_file.stat().st_size / (1024 * 1024)  # MB
    print(f"   • {csv_file.name} ({size:.1f} MB)")

📂 Dossier CSV: /home/esprit/airlLines_Project/dataSetAirlines
📄 Nombre de fichiers: 11

Fichiers disponibles:
   • Airline_Scrapped_Review.csv (3.4 MB)
   • Calendar.csv (0.1 MB)
   • Customer_Flight_Activity.csv (10.6 MB)
   • Customer_Loyalty_History.csv (1.7 MB)
   • Customer_comment.csv (3.2 MB)
   • Passanger_booking_data.csv (3.0 MB)
   • Survey_data_Inflight_Satisfaction_Score.csv (13.9 MB)
   • airline_passenger_satisfaction.csv (12.3 MB)
   • airline_ticket_prices_dataset.csv (0.0 MB)
   • airlines_flights_data.csv (23.8 MB)
   • airlines_reviews.csv (7.6 MB)


## 6. Test d'extraction SQL (airlines_dwh) - FIX: SQLAlchemy

In [15]:
# Création d'un engine SQLAlchemy (recommandé pour pandas)
try:
    # Connection string pour SQL Server
    connection_string = f"mssql+pymssql://{sql_config['user']}:{sql_config['password']}@{sql_config['server']}:{sql_config['port']}/{sql_config['databases']['dwh']}"
    engine = create_engine(connection_string)
    
    query = """
    SELECT TOP 5 
        review_sk, satisfaction_status,
        seat_comfort_score, food_drink_score,
        departure_delay, arrival_delay
    FROM FACT_REVIEW
    """
    
    df_sample = pd.read_sql(query, engine)
    
    print("✅ Extraction SQL réussie (via SQLAlchemy - PAS DE WARNING)")
    print(f"📊 Shape: {df_sample.shape}")
    print("\nAperçu des données:")
    display(df_sample)
    
except Exception as e:
    print(f"❌ Erreur extraction SQL: {e}")

✅ Extraction SQL réussie (via SQLAlchemy - PAS DE WARNING)
📊 Shape: (5, 6)

Aperçu des données:


,review_sk,satisfaction_status,seat_comfort_score,food_drink_score,departure_delay,arrival_delay
0,1,Neutral or Dissatisfied,5,5,2,5
1,2,Satisfied,4,3,26,39
2,3,Satisfied,5,5,0,0
3,4,Satisfied,5,4,0,0
4,5,Satisfied,4,4,0,1


## 7. Test lecture CSV

In [16]:
# Lecture d'un fichier CSV
csv_file = csv_dir / "airline_passenger_satisfaction.csv"

if csv_file.exists():
    df_csv = pd.read_csv(csv_file, nrows=5)
    print("✅ Lecture CSV réussie")
    print(f"📄 Fichier: {csv_file.name}")
    print(f"📊 Colonnes: {list(df_csv.columns)}")
    print("\nAperçu:")
    display(df_csv)
else:
    print(f"❌ Fichier non trouvé: {csv_file}")

✅ Lecture CSV réussie
📄 Fichier: airline_passenger_satisfaction.csv
📊 Colonnes: ['ID', 'Gender', 'Age', 'Customer Type', 'Type of Travel', 'Class', 'Flight Distance', 'Departure Delay', 'Arrival Delay', 'Departure and Arrival Time Convenience', 'Ease of Online Booking', 'Check-in Service', 'Online Boarding', 'Gate Location', 'On-board Service', 'Seat Comfort', 'Leg Room Service', 'Cleanliness', 'Food and Drink', 'In-flight Service', 'In-flight Wifi Service', 'In-flight Entertainment', 'Baggage Handling', 'Satisfaction']

Aperçu:


,ID,Gender,Age,Customer Type,Type of Travel,Class,Flight Distance,Departure Delay,Arrival Delay,Departure and Arrival Time Convenience,...,On-board Service,Seat Comfort,Leg Room Service,Cleanliness,Food and Drink,In-flight Service,In-flight Wifi Service,In-flight Entertainment,Baggage Handling,Satisfaction
0,1,Male,48,First-time,Business,Business,821,2,5,3,...,3,5,2,5,5,5,3,5,5,Neutral or Dissatisfied
1,2,Female,35,Returning,Business,Business,821,26,39,2,...,5,4,5,5,3,5,2,5,5,Satisfied
2,3,Male,41,Returning,Business,Business,853,0,0,4,...,3,5,3,5,5,3,4,3,3,Satisfied
3,4,Male,50,Returning,Business,Business,1905,0,0,2,...,5,5,5,4,4,5,2,5,5,Satisfied
4,5,Female,49,Returning,Business,Business,3470,0,1,3,...,3,4,4,5,4,3,3,3,3,Satisfied


## 8. Vérification environnement Python

In [17]:
print("🐍 Environnement Python:")
print(f"   Version: {sys.version}")
print(f"   Chemin: {sys.executable}")
print(f"   Conda env: {os.getenv('CONDA_DEFAULT_ENV', 'N/A')}")

print("\n📦 Versions des packages:")
print(f"   pandas: {pd.__version__}")
print(f"   numpy: {np.__version__}")
print(f"   matplotlib: {plt.matplotlib.__version__}")
print(f"   pymssql: {pymssql.__version__}")

🐍 Environnement Python:
   Version: 3.11.15 (main, Mar 11 2026, 17:20:07) [GCC 14.3.0]
   Chemin: /home/esprit/miniconda3/envs/airlines/bin/python3.11
   Conda env: airlines

📦 Versions des packages:
   pandas: 3.0.3
   numpy: 2.4.6
   matplotlib: 3.10.9
   pymssql: 2.3.13
